In [1]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

matplotlib.rcParams["font.family"] = "Linux Libertine O"
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
dirbase = "figures/eval/"
ourSys = "EinNet"

sz, fontsz = (6, 3), 16
figsz = {
    "axes.labelsize": 14,
    "font.size": 14,
    "legend.fontsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.figsize": (6, 3),
    "axes.titlesize": 14,
}
plt.rcParams.update(figsz)

hb = "\\\\//\\\\//"

color_def = [
    "#f4b183",
    "#ffd966",
    "#c5e0b4",
    "#bdd7ee",
    "#8dd3c7",
    "#bebada",
    "#fb8072",
    "#80b1d3",
    "#fdb462",
    "#cccccc",
    "#fccde5",
    "#b3de69",
    "#ffd92f",
    "#fc8d59",
    "#74a9cf",
    "#66c2a4",
    "#f4a143",
    "#ffc936",
    "#78c679",
]

color_line = [
    "#5AA469",
    "#1A508B",
    "#F39233",
    "#B61919",
]

hatch_def = [
    "//",
    "\\\\",
    "xx",
    "++",
    "--",
    "||",
    "..",
    "oo",
    "",
]

marker_def = [
    "o",
    "x",
    "D",
    "*",
    "+",
]


In [17]:
data_set = pd.read_parquet(
    "/home/zly/Works/uniserving/exp/dataset/data_diffusiondb/metadata-large.parquet", engine="fastparquet" # Unique ratio: num_unique/num_all=0.1
    # "/home/zly/Works/uniserving/exp/dataset/data_diffusiondb/metadata.parquet", engine="fastparquet" # Unique ratio: num_unique/num_all=0.764257
)
data_set


,image_name,prompt,part_id,seed,step,cfg,sampler,width,height,user_name,timestamp,image_nsfw,prompt_nsfw
0,3ccdc650-871a-4ad9-9bf2-dc475b83ed32.webp,beautiful porcelain ivory fair face woman biom...,1,2625978990,50,21.0,8,512,704,01f4e782b48faedf416083b2fbabaca2a45621b15ead23...,2022-08-20 10:03:00+00:00,0.038466,0.003089
1,1f1fcb70-63a4-40b1-ada9-2c15fb2ca10a.webp,complex 3 d render hyper detailed ultra sharp ...,1,738462306,50,10.0,8,512,704,01f4e782b48faedf416083b2fbabaca2a45621b15ead23...,2022-08-20 10:55:00+00:00,0.187317,0.001722
2,b0809c6b-cf43-4a82-99f7-6f2947d433fc.webp,complex 3 d render hyper detailed ultra sharp ...,1,1584972414,50,10.0,8,512,704,01f4e782b48faedf416083b2fbabaca2a45621b15ead23...,2022-08-20 10:55:00+00:00,0.065495,0.001722
3,b8cff57e-eb9d-467a-95a1-f6e3b8a38575.webp,complex 3 d render hyper detailed ultra sharp ...,1,2816373313,50,10.0,8,512,704,01f4e782b48faedf416083b2fbabaca2a45621b15ead23...,2022-08-20 10:55:00+00:00,0.083114,0.001722
4,298086cb-1c05-424e-b83b-a6148e8816e2.webp,complex 3 d render hyper detailed ultra sharp ...,1,3079866895,50,10.0,8,512,704,01f4e782b48faedf416083b2fbabaca2a45621b15ead23...,2022-08-20 10:55:00+00:00,0.148977,0.001722
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13999995,fb6ced8d-4627-45d2-8710-e5216c0b794e.webp,"Ibai Berto Romero as Willy Wonka, highly detai...",14000,2203538431,50,7.0,8,512,768,fcdb3e09f977412c342b6624a19d1295ee1334c153c90a...,2022-08-07 22:57:00+00:00,0.052320,0.000555
13999996,396e544e-344b-4971-a081-8d58936e17c2.webp,"Ibai Berto Romero as Willy Wonka, highly detai...",14000,1625163903,50,7.0,8,512,768,fcdb3e09f977412c342b6624a19d1295ee1334c153c90a...,2022-08-07 22:57:00+00:00,0.117837,0.000555
13999997,78fa6f35-9468-40ce-95ff-08567b5740d8.webp,"Ibai Berto Romero as Willy Wonka, highly detai...",14000,1987919615,50,7.0,8,512,768,fcdb3e09f977412c342b6624a19d1295ee1334c153c90a...,2022-08-07 22:57:00+00:00,0.069527,0.000555
13999998,383380d9-8768-41d6-8938-d6f67a68b367.webp,"Ibai Berto Romero as Willy Wonka, highly detai...",14000,1880276095,50,7.0,8,512,768,fcdb3e09f977412c342b6624a19d1295ee1334c153c90a...,2022-08-07 22:57:00+00:00,0.050364,0.000555


In [21]:
# full_dataset = data_set
# print(len(data_set['seed']))
n_tot = len(data_set['seed'])
n_unique = len(data_set['seed'].unique())
print(f"{n_unique/n_tot} {n_unique}/{n_tot}")
n_once = (~data_set['seed'].duplicated(keep=False)).sum()
# n_once = len(out)
print(f"{n_once/n_tot} {n_once}/{n_tot}")

0.8962212857142857 12547098/14000000
1.0 14000000/14000000


In [62]:
data_set = full_dataset


In [63]:
def find_cacndiate_example():
    k = 5
    keys = ["step", "cfg", "width", "height"]
    for i in range(14000000):
        if i % 100000 == 0:
            print('step', i)
        d = [False] * 4
        for j in range(i, i+k):
            if data_set.iloc[j]["prompt"] == data_set.iloc[j + 1]["prompt"] and not all(
                [
                    data_set.iloc[j][k] == data_set.iloc[j + 1][k]
                    for k in keys
                ]
            ):
                for idx, key  in enumerate(keys):
                    d[idx] = d[idx] or data_set.iloc[j][key] != data_set.iloc[j + 1][key]
                continue
            else:
                break
        else:
            # print(i, d)
            if (d[0] or d[1]) and (d[2] or d[3]) and data_set.iloc[i:i+k]['cfg'].max()<13:
                print(data_set.iloc[i:i+k])
# result: 2518044, 2518045, 3700057, 4224372, 4224373, 4224374, 4314413, 4324068, 4324069, 4324070, 4324071, 4485832, 4791260, 5314234, 6934929, 7107859, 7487246, 7740451, 8001208, 8001209, 8001210, 8001211, 8001212, 8303013, 8303014, 8303015, 8303016, 8640374, 8835685, 9534721, 9592759, 9592760, 10115027, 10115028, 10115029, 10125943, 10125944, 11141752, 11141753, 11141754, 11378434, 11662958, 11662959, 11673610, 11980035, 11991662, 11991663, 11991664, 11991665, 12004933, 12268688, 12650452, 12705215, 12963293, 12963294, 13164992, 13164993, 13164994, 13179882, 13917522, 13926641


In [ ]:
print(data_set.iloc[11141752]['prompt'])


two young men, cute handsome beautiful dark medium wavy hair man in his 2 0 s named shadow taehyung and young cute handsome dark red medium length curly hair man named maximo together at the halloween party, elegant, wearing suits!, modest!, delicate facial features, art by alphonse mucha, vincent van gogh, egon schiele 


In [64]:
def calc_ratio(data_set):
    prompt_all=data_set['prompt']
    prompt_unique=prompt_all.unique()
    num_all=len(prompt_all)
    num_unique=len(prompt_unique)
    print(f'{num_all=} {num_unique=} {num_unique/num_all=}')
    return prompt_all
prompt_all = calc_ratio(data_set)


num_all=14000000 num_unique=1819808 num_unique/num_all=0.12998628571428572


In [65]:
prompt_all


0           beautiful porcelain ivory fair face woman biom...
1           complex 3 d render hyper detailed ultra sharp ...
2           complex 3 d render hyper detailed ultra sharp ...
3           complex 3 d render hyper detailed ultra sharp ...
4           complex 3 d render hyper detailed ultra sharp ...
                                  ...                        
13999995    Ibai Berto Romero as Willy Wonka, highly detai...
13999996    Ibai Berto Romero as Willy Wonka, highly detai...
13999997    Ibai Berto Romero as Willy Wonka, highly detai...
13999998    Ibai Berto Romero as Willy Wonka, highly detai...
13999999                              paella by studio ghibli
Name: prompt, Length: 14000000, dtype: object

In [66]:
counts = prompt_all.value_counts()
# del data_set
counts


prompt
wrc rally car stylize, art gta 5 cover, official fanart behance hd artstation by jesper ejsing, by rhads, makoto shinkai and lois van baarle, ilya kuvshinov, ossdraws, that looks like it is from borderlands and by feng zhu and loish and laurie greasley, victo ngai, andreas rocha, john harris                                                                                                        5410
epic mask helmet robot ninja portrait stylized as fornite style game design fanart by concept artist gervasio canda, behance hd by jesper ejsing, by rhads, makoto shinkai and lois van baarle, ilya kuvshinov, rossdraws global illumination radiating a glowing aura global illumination ray tracing hdr render in unreal engine 5                                                                        4113
photo wallpaper sport car gran turismo 7 forza horizon need for speed fast and furious 5 unreal engine supercar hypercar game concept car octane render, 4 khd 2 0 2 2 3 d cgi rtx style chrome

In [67]:
cnt_occurrence=counts.groupby(counts).count()
cnt_occurrence


count
1       250727
2        19009
3        24115
4       411634
5        13834
         ...  
3474         1
3771         1
4095         1
4113         1
5410         1
Name: count, Length: 451, dtype: int64

In [68]:
# group_occurrence = cnt_occurrence.groupby(pd.cut(cnt_occurrence.index, [0, 1, 5, 10, 10000])).sum()
group_occurrence = cnt_occurrence.groupby(pd.cut(cnt_occurrence.index, [0, 1, 4, 8, 16, 10000])).sum()
group_occurrence


/tmp/ipykernel_2659995/1866886176.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_occurrence = cnt_occurrence.groupby(pd.cut(cnt_occurrence.index, [0, 1, 4, 8, 16, 10000])).sum()


(0, 1]         250727
(1, 4]         454758
(4, 8]         261376
(8, 16]        742857
(16, 10000]    110090
Name: count, dtype: int64

In [60]:
cnt_occurrence.values


array([2, 1, 1])

In [61]:
cnt_queries = cnt_occurrence.index.astype('int') * cnt_occurrence.values
cnt_queries = cnt_queries.to_series(index = cnt_occurrence.index)
# cnt_occurrence.values

group_occurrence = cnt_queries.groupby(pd.cut(cnt_queries.index, [0, 1, 5, 10, 10000])).sum()
group_occurrence


/tmp/ipykernel_2659995/551901643.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_occurrence = cnt_queries.groupby(pd.cut(cnt_queries.index, [0, 1, 5, 10, 10000])).sum()


(0, 1]          2
(1, 5]          4
(5, 10]         0
(10, 10000]    14
Name: count, dtype: int64

In [ ]:
def draw_pie_chart(keys, values):
    # TODO: plot the figure with correct labels
    fig, ax = plt.subplots()

    ax.pie(values, labels=keys, autopct='%1.1f%%', colors=color_def, textprops={'fontsize': 16})

    # series.plot(kind='pie', autopct='%1.1f%%', label)
    plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
    plt.savefig("figures/pie.pdf", bbox_inches='tight')
    plt.show()

draw_pie_chart(['1', '2-5', '6-10', '>10'], group_occurrence.values)


In [7]:
prompt_all.str.len().describe()


count    1.400000e+07
mean     1.703520e+02
std      1.145286e+02
min      0.000000e+00
25%      7.600000e+01
50%      1.520000e+02
75%      2.490000e+02
max      1.923000e+03
Name: prompt, dtype: float64

In [8]:
prompt_tokens = prompt_all.apply(lambda x: len(x.split()))
prompt_tokens.describe()


count    1.400000e+07
mean     2.570424e+01
std      1.664061e+01
min      0.000000e+00
25%      1.200000e+01
50%      2.300000e+01
75%      3.700000e+01
max      4.800000e+02
Name: prompt, dtype: float64

In [11]:
prompt_unique_tokens = pd.Series(prompt_unique).apply(lambda x: len(x.split()))
prompt_unique_tokens.describe()


count    1.819808e+06
mean     2.314020e+01
std      1.589169e+01
min      0.000000e+00
25%      1.000000e+01
50%      2.000000e+01
75%      3.400000e+01
max      4.800000e+02
dtype: float64